# End-to-End: Train on the Conversation, Test on Unknown Text

This notebook runs the full pipeline on a corpus built from **this conversation**
(`ewm-cortex/corpus/conversation.txt`):

1. **Data** — character-level dataset, split 90% train / 10% held-out test.
2. **Phase 0** — learned-K baseline: train on the train split, evaluate on the test split.
3. **Phase 1** — K-storage attach: coverage invariant + collision statistics.
4. **Phase 2** — address-key attention (`k_t = E_bit[bit(t)]`): train, evaluate on the
   test split, compare the gap against the baseline, and run the hybrid gate.
5. **Phase 3** — MoE / Expert Think Tank / resolution (A+B) over the conversation
   sentences.

> **evcxr note.** Path-dependency warnings from `hllset-next` are non-fatal. Cells use
> owned data (`.to_vec()`, `.to_string()`) and block scopes so no borrow crosses a
> cell boundary. If the dep warnings bother you, launch Jupyter with
> `RUSTFLAGS="--cap-lints=allow"`.


In [2]:
:dep hllset-core = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/crates/hllset-core" }
:dep hllset-attn = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/crates/hllset-attn" }
:dep hllset-repro = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/crates/hllset-repro" }

use hllset_repro::{Adam, CharDataset, Config, Transformer, XorShift, attach, bits_for_dataset, run_phase3};
use hllset_attn::*;
use hllset_core::HLLSet;

let corpus = std::fs::read_to_string(
    "/home/alexmy/SGS/SGS_lib/fractal_manifold/ewm-cortex/corpus/conversation.txt"
).expect("corpus file");
println!("corpus loaded: {} chars", corpus.len());


corpus loaded: 4337 chars


---
## 1. Data — one vocabulary, held-out test split

A single dataset is built over the whole corpus so the vocabulary is complete;
only the *token ids* are split. The test portion is **unknown text**: it is
never used for training or K-storage instrumentation statistics beyond the
shared vocabulary.


In [3]:
let full_ds = CharDataset::from_text(&corpus);
let split = full_ds.data().len() * 9 / 10;
let train_data = full_ds.data()[..split].to_vec(); // owned
let test_data = full_ds.data()[split..].to_vec();  // owned
println!("vocab     : {} chars", full_ds.vocab_size());
println!("train ids : {}  (windows for block 32: {})", train_data.len(), train_data.len() - 33);
println!("test  ids : {}  (held out, never trained on)", test_data.len());


vocab     : 41 chars
train ids : 3903  (windows for block 32: 3870)
test  ids : 434  (held out, never trained on)


---
## 2. Phase 0 — learned-K baseline


In [4]:
let mut baseline_test_loss: f32 = 0.0;
{
    let cfg = Config::small(full_ds.vocab_size(), 32);
    let model = Transformer::new(cfg, 42);
    let mut opt = Adam::new(1e-3);
    let mut rng = XorShift::new(7);
    let max_start = train_data.len() - 33;
    for step in 0..500 {
        let start = rng.index(max_start);
        let inputs = train_data[start..start + 32].to_vec();
        let targets = train_data[start + 1..start + 33].to_vec();
        let loss = model.train_step(&mut opt, &inputs, &targets);
        if (step + 1) % 125 == 0 {
            println!("  step {:>4}: train loss {:.4}", step + 1, loss);
        }
    }
    baseline_test_loss = model.eval_loss(&test_data, 32);
    println!("  baseline test loss (unknown text): {:.4}", baseline_test_loss);
}


  step  125: train loss 2.8337
  step  250: train loss 2.2448
  step  375: train loss 2.0709
  step  500: train loss 2.4543
  baseline test loss (unknown text): 2.3342


()

---
## 3. Phase 1 — K-storage attach


In [5]:
let report = attach(&full_ds);
println!("coverage (invariant) : {:.3} {}", report.coverage, if report.coverage == 1.0 { "✓" } else { "✗" });
println!("corpus collisions    : {} observed pairs (expected {:.2})",
    report.corpus.observed_pairs, report.corpus.expected_pairs);
println!("synthetic 5000-token : {} observed pairs vs {:.2} expected — {}",
    report.synthetic.observed_pairs, report.synthetic.expected_pairs,
    if report.synthetic_matches_theory { "✓ matches 1/3072" } else { "✗" });


coverage (invariant) : 1.000 ✓
corpus collisions    : 1 observed pairs (expected 0.27)
synthetic 5000-token : 3942 observed pairs vs 4068.20 expected — ✓ matches 1/3072


---
## 4. Phase 2 — address-key attention + hybrid gate


In [6]:
let mut address_test_loss: f32 = 0.0;
{
    let bits = bits_for_dataset(&full_ds);
    let cfg = Config::small(full_ds.vocab_size(), 32);
    let model = Transformer::new_address(cfg, 42, &bits);
    let mut opt = Adam::new(1e-3);
    let mut rng = XorShift::new(7);
    let max_start = train_data.len() - 33;
    for step in 0..500 {
        let start = rng.index(max_start);
        let inputs = train_data[start..start + 32].to_vec();
        let targets = train_data[start + 1..start + 33].to_vec();
        let loss = model.train_step(&mut opt, &inputs, &targets);
        if (step + 1) % 125 == 0 {
            println!("  step {:>4}: train loss {:.4}", step + 1, loss);
        }
    }
    address_test_loss = model.eval_loss(&test_data, 32);
    let gap = (address_test_loss - baseline_test_loss) / baseline_test_loss;
    println!("  address test loss (unknown text): {:.4}", address_test_loss);
    println!("  gap vs baseline: {:+.2}% {}", gap * 100.0, if gap <= 0.05 { "✓ within 5%" } else { "✗" });

    // ── Mode C hybrid gate: logits + λ · 1[bit ∈ F(t)] ──
    let char_obs: Vec<HLLSet> = corpus.lines().filter(|l| !l.trim().is_empty())
        .map(|l| {
            let chars: Vec<Vec<u8>> = l.chars().map(|c| c.to_string().into_bytes()).collect();
            HLLSet::from_tokens(chars.iter())
        })
        .collect();
    let recent_start = char_obs.len().saturating_sub(2);
    let mut recent = HLLSet::new();
    for obs in &char_obs[recent_start..] { recent.merge(obs); }
    let moe = MoE::from_observations(&char_obs, 3, 2, None);
    let resolved = moe.resolve(&recent, 0.3);

    let prompt = full_ds.encode("The next phase");
    let logits = model.forward(&prompt);
    let logits_data = logits.borrow().data.clone();
    let n = full_ds.vocab_size();
    let base: Vec<f32> = logits_data[(prompt.len() - 1) * n..].to_vec();
    for lambda in [0.0f32, 2.0] {
        let mut gated = base.clone();
        for (tok, &bit) in bits.iter().enumerate() {
            if resolved.bitmap().contains(bit) { gated[tok] += lambda; }
        }
        let mut idx: Vec<usize> = (0..gated.len()).collect();
        idx.sort_by(|&a, &b| gated[b].partial_cmp(&gated[a]).unwrap());
        idx.truncate(5);
        let top: Vec<String> = idx.iter().map(|&t| format!("{} ({:.2})", full_ds.decode(&[t]), gated[t])).collect();
        println!("  gate λ={lambda:.1}: {}", top.join(", "));
    }
}


  step  125: train loss 2.8952
  step  250: train loss 2.2263
  step  375: train loss 2.0424
  step  500: train loss 2.5737
  address test loss (unknown text): 2.3449
  gap vs baseline: +0.46% ✓ within 5%
  gate λ=0.0:   (4.16), r (2.16), d (2.09), s (1.86), . (1.48)
  gate λ=2.0:   (6.16), r (4.16), d (4.09), s (3.86), . (3.48)


()

---
## 5. Phase 3 — MoE / Expert Think Tank / resolution (A+B)


In [7]:
let report3 = run_phase3(&corpus, 3, 2, 0.3, 2);
println!("observations : {}", report3.observations);
println!("experts      : {}", report3.experts);
println!("leader ρ     : {:.3}", report3.leader_relevance);
println!("F(t)         : {} bits, {} tokens, coverage {:.2}",
    report3.resolved_bits, report3.resolved_tokens, report3.coverage);
println!("top tokens (output ranking, resolution B):");
for (token, count) in &report3.top_tokens {
    println!("  {token:>12} ×{count}");
}


observations : 11
experts      : 6
leader ρ     : 0.652
F(t)         : 202 bits, 220 tokens, coverage 1.00
top tokens (output ranking, resolution B):
           the ×37
           The ×19
            of ×12
            is ×11
           and ×10
             a ×7
    vocabulary ×5
       context ×4
          gate ×4
       lattice ×4


()

---
## Summary

The whole arc runs on one corpus built from this conversation:

- **Phase 0/2** — the learned-K baseline and the address-key model both train
  on the train split and are evaluated on the **held-out test split**; the
  address keys (HLLSet bit cells as K) reproduce the baseline within the 5%
  exit criterion.
- **Phase 1** — coverage invariant holds; observed collisions match `1/3072`.
- **Phase 3** — the SHA1-shuffled MoE produces the Expert Think Tank, the
  Expert Leader, and the resolved `F(t)`; output ranking shows
  what the leader says first.
- **Mode C gate** — the resolved context `F(t)` gates the transformer's
  next-token scores.
